In [1]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..', '..')))

import argparse
from pprint import pp
import torch
from torch import nn
from tqdm import tqdm
import numpy as np
import json
import os
from omegaconf import OmegaConf
from torch.utils.tensorboard import SummaryWriter
from torchinfo import summary

from lpn.utils import load_dataset, load_config
from lpn.utils import get_model
from lpn.utils import get_loss_hparams_and_lr, get_loss
from lpn.utils import trainer
from lpn.utils import utils
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as skimage_psnr
from skimage.metrics import structural_similarity as skimage_ssim

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [3]:
lpn_model_config_path = os.path.join('..','models','lpn','s=0.1','model_config.json')
lpn_model_weight_path = os.path.join('..','models','lpn','s=0.1','model.pt')

lpn_model_config = load_config(lpn_model_config_path)
lpn_model = get_model(lpn_model_config)
lpn_model.load_state_dict(torch.load(lpn_model_weight_path)['model_state_dict'])
lpn_model.to(device)

init weights


LPN(
  (lin): ModuleList(
    (0): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (3): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (5): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (6): Conv2d(256, 64, kernel_size=(16, 16), stride=(1, 1), bias=False)
    (7): Linear(in_features=64, out_features=1, bias=True)
  )
  (res): ModuleList(
    (0): Conv2d(3, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (2): Conv2d(3, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (3): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 

In [4]:
ne_lpn_model_config_path = os.path.join('..','models','ne_lpn','s=0.1','model_config.json')
ne_lpn_model_weight_path = os.path.join('..','models','ne_lpn','s=0.1','model.pt')
ne_lpn_model_config = load_config(ne_lpn_model_config_path)
ne_lpn_model = get_model(ne_lpn_model_config)
ne_lpn_model.load_state_dict(torch.load(ne_lpn_model_weight_path)['model_state_dict'])
ne_lpn_model.to(device)

init weights


LPN(
  (lin): ModuleList(
    (0): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (3): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (5): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (6): Conv2d(256, 64, kernel_size=(16, 16), stride=(1, 1), bias=False)
    (7): Linear(in_features=64, out_features=1, bias=False)
  )
  (res): ModuleList(
    (0): Conv2d(3, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (2): Conv2d(3, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (3): Conv2d(3, 256

In [5]:
ne_lpn_by_forward_model_config_path = '../models/ne_by_forward_lpn/s=0.1/model_config.json'
ne_lpn_by_forward_model_weight_path = '../models/ne_by_forward_lpn/s=0.1/model.pt'
ne_lpn_by_forward_model_config = load_config(ne_lpn_by_forward_model_config_path)
ne_lpn_by_forward_model = get_model(ne_lpn_by_forward_model_config)
ne_lpn_by_forward_model.load_state_dict(torch.load(ne_lpn_by_forward_model_weight_path)['model_state_dict'])
ne_lpn_by_forward_model.to(device)

init weights


LPN(
  (lin): ModuleList(
    (0): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (3): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (5): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (6): Conv2d(256, 64, kernel_size=(16, 16), stride=(1, 1), bias=False)
    (7): Linear(in_features=64, out_features=1, bias=True)
  )
  (res): ModuleList(
    (0): Conv2d(3, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (2): Conv2d(3, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (3): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 

In [6]:
ne_lpn_normalized_and_scaled_model_config_path = '../models/ne_norm_and_scaled_lpn/s=0.1/model_config.json'
ne_lpn_normalized_and_scaled_model_weight_path = '../models/ne_norm_and_scaled_lpn/s=0.1/model.pt'
ne_lpn_normalized_and_scaled_model_config = load_config(ne_lpn_normalized_and_scaled_model_config_path)
ne_lpn_normalized_and_scaled_model = get_model(ne_lpn_normalized_and_scaled_model_config)
ne_lpn_normalized_and_scaled_model.load_state_dict(torch.load(ne_lpn_normalized_and_scaled_model_weight_path)['model_state_dict'])
ne_lpn_normalized_and_scaled_model.to(device)

init weights


LPN(
  (lin): ModuleList(
    (0): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (3): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (5): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (6): Conv2d(256, 64, kernel_size=(16, 16), stride=(1, 1), bias=False)
    (7): Linear(in_features=64, out_features=1, bias=False)
  )
  (res): ModuleList(
    (0): Conv2d(3, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (2): Conv2d(3, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (3): Conv2d(3, 256

In [7]:
drunet_model_config_path = '../models/drunet/s=0.1/model_config.json'
drunet_model_weight_path = '../models/drunet/s=0.1/model.pt'
drunet_model_config = load_config(drunet_model_config_path)
drunet_model = get_model(drunet_model_config)
#drunet_model.load_state_dict(torch.load(drunet_model_weight_path)['model_state_dict'])
drunet_model.to(device)

DRUNet(
  (m_head): AffineConv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
  (m_down): ModuleList(
    (0): Sequential(
      (0): ResBlock(
        (m_res): Sequential(
          (0): AffineConv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
          (1): SortPool()
          (2): AffineConv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
        )
        (sum): ResidualConnection()
      )
      (1): ResBlock(
        (m_res): Sequential(
          (0): AffineConv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
          (1): SortPool()
          (2): AffineConv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
        )
        (sum): ResidualConnection()
      )
      (2): ResBlock(
        (m_res): Sequential(
          (0): AffineConv2d(64, 64, 

In [8]:
in_dim = 3 # because these are RGB images, change to 1 for grayscale testing

In [9]:
# First, check the params of the U-Net model

summary(drunet_model, input_size=(1,in_dim,128,128))

Layer (type:depth-idx)                        Output Shape              Param #
DRUNet                                        [1, 3, 128, 128]          --
├─AffineConv2d: 1-1                           [1, 64, 128, 128]         1,728
├─ModuleList: 1-2                             --                        --
│    └─Sequential: 2-1                        [1, 128, 64, 64]          --
│    │    └─ResBlock: 3-1                     [1, 64, 128, 128]         73,729
│    │    └─ResBlock: 3-2                     [1, 64, 128, 128]         73,729
│    │    └─ResBlock: 3-3                     [1, 64, 128, 128]         73,729
│    │    └─ResBlock: 3-4                     [1, 64, 128, 128]         73,729
│    │    └─AffineConv2d: 3-5                 [1, 128, 64, 64]          32,768
│    └─Sequential: 2-2                        [1, 256, 32, 32]          --
│    │    └─ResBlock: 3-6                     [1, 128, 64, 64]          294,913
│    │    └─ResBlock: 3-7                     [1, 128, 64, 64]     

In [10]:
# Now, the LPN model

summary(lpn_model, input_size=(1,in_dim,128,128))

Layer (type:depth-idx)                   Output Shape              Param #
LPN                                      [1, 3, 128, 128]          --
├─ModuleList: 1-21                       --                        (recursive)
│    └─Conv2d: 2-1                       [1, 256, 128, 128]        7,168
├─Softplus: 1-2                          [1, 256, 128, 128]        --
├─ModuleList: 1-21                       --                        (recursive)
│    └─Conv2d: 2-2                       [1, 256, 64, 64]          589,824
├─ModuleList: 1-19                       --                        (recursive)
│    └─Conv2d: 2-3                       [1, 256, 64, 64]          7,168
├─Softplus: 1-5                          [1, 256, 64, 64]          --
├─ModuleList: 1-21                       --                        (recursive)
│    └─Conv2d: 2-4                       [1, 256, 64, 64]          589,824
├─ModuleList: 1-19                       --                        (recursive)
│    └─Conv2d: 2-5      

In [11]:
# Now, the NE-LPN model

summary(ne_lpn_model, input_size=(1,in_dim,128,128))

Layer (type:depth-idx)                   Output Shape              Param #
LPN                                      [1, 3, 128, 128]          --
├─ModuleList: 1-21                       --                        (recursive)
│    └─Conv2d: 2-1                       [1, 256, 128, 128]        6,912
├─SortPool: 1-2                          [1, 256, 128, 128]        --
├─ModuleList: 1-21                       --                        (recursive)
│    └─Conv2d: 2-2                       [1, 256, 64, 64]          589,824
├─ModuleList: 1-19                       --                        (recursive)
│    └─Conv2d: 2-3                       [1, 256, 64, 64]          6,912
├─SortPool: 1-5                          [1, 256, 64, 64]          --
├─ModuleList: 1-21                       --                        (recursive)
│    └─Conv2d: 2-4                       [1, 256, 64, 64]          589,824
├─ModuleList: 1-19                       --                        (recursive)
│    └─Conv2d: 2-5      

In [12]:
# Finally, the NE-LPN by forward model

summary(ne_lpn_by_forward_model, input_size=(1,in_dim,128,128))

Layer (type:depth-idx)                   Output Shape              Param #
LPN                                      [1, 3, 128, 128]          --
├─ModuleList: 1-21                       --                        (recursive)
│    └─Conv2d: 2-1                       [1, 256, 128, 128]        7,168
├─Softplus: 1-2                          [1, 256, 128, 128]        --
├─ModuleList: 1-21                       --                        (recursive)
│    └─Conv2d: 2-2                       [1, 256, 64, 64]          589,824
├─ModuleList: 1-19                       --                        (recursive)
│    └─Conv2d: 2-3                       [1, 256, 64, 64]          7,168
├─Softplus: 1-5                          [1, 256, 64, 64]          --
├─ModuleList: 1-21                       --                        (recursive)
│    └─Conv2d: 2-4                       [1, 256, 64, 64]          589,824
├─ModuleList: 1-19                       --                        (recursive)
│    └─Conv2d: 2-5      

In [13]:
# Now, NE-LPN with normalization in forward and the squaring of output

summary(ne_lpn_normalized_and_scaled_model, input_size=(1,in_dim,128,128))

Layer (type:depth-idx)                   Output Shape              Param #
LPN                                      [1, 3, 128, 128]          --
├─ModuleList: 1-21                       --                        (recursive)
│    └─Conv2d: 2-1                       [1, 256, 128, 128]        6,912
├─SortPool: 1-2                          [1, 256, 128, 128]        --
├─ModuleList: 1-21                       --                        (recursive)
│    └─Conv2d: 2-2                       [1, 256, 64, 64]          589,824
├─ModuleList: 1-19                       --                        (recursive)
│    └─Conv2d: 2-3                       [1, 256, 64, 64]          6,912
├─SortPool: 1-5                          [1, 256, 64, 64]          --
├─ModuleList: 1-21                       --                        (recursive)
│    └─Conv2d: 2-4                       [1, 256, 64, 64]          589,824
├─ModuleList: 1-19                       --                        (recursive)
│    └─Conv2d: 2-5      

In [14]:
drunet_mini_model_config_path = '../models/drunet_mini/s=0.1/model_config.json'
drunet_mini_model_weight_path = '../models/drunet_mini/s=0.1/model.pt'
drunet_mini_model_config = load_config(drunet_mini_model_config_path)
drunet_mini_model = get_model(drunet_mini_model_config)
drunet_mini_model.load_state_dict(torch.load(drunet_mini_model_weight_path)['model_state_dict'])
drunet_mini_model.to(device)

DRUNet(
  (m_head): AffineConv2d(3, 34, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
  (m_down): ModuleList(
    (0): Sequential(
      (0): ResBlock(
        (m_res): Sequential(
          (0): AffineConv2d(34, 34, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
          (1): SortPool()
          (2): AffineConv2d(34, 34, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
        )
        (sum): ResidualConnection()
      )
      (1): ResBlock(
        (m_res): Sequential(
          (0): AffineConv2d(34, 34, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
          (1): SortPool()
          (2): AffineConv2d(34, 34, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
        )
        (sum): ResidualConnection()
      )
      (2): ResBlock(
        (m_res): Sequential(
          (0): AffineConv2d(34, 34, 

In [15]:
summary(drunet_mini_model, input_size=(1,in_dim,128,128))

Layer (type:depth-idx)                        Output Shape              Param #
DRUNet                                        [1, 3, 128, 128]          --
├─AffineConv2d: 1-1                           [1, 34, 128, 128]         918
├─ModuleList: 1-2                             --                        --
│    └─Sequential: 2-1                        [1, 68, 64, 64]           --
│    │    └─ResBlock: 3-1                     [1, 34, 128, 128]         20,809
│    │    └─ResBlock: 3-2                     [1, 34, 128, 128]         20,809
│    │    └─ResBlock: 3-3                     [1, 34, 128, 128]         20,809
│    │    └─AffineConv2d: 3-4                 [1, 68, 64, 64]           9,248
│    └─Sequential: 2-2                        [1, 136, 32, 32]          --
│    │    └─ResBlock: 3-5                     [1, 68, 64, 64]           83,233
│    │    └─ResBlock: 3-6                     [1, 68, 64, 64]           83,233
│    │    └─ResBlock: 3-7                     [1, 68, 64, 64]          

In [16]:
less_drunet_mini_model_config_path = '../models/less_drunet_mini/s=0.1/model_config.json'
less_drunet_mini_model_weight_path = '../models/less_drunet_mini/s=0.1/model.pt'
less_drunet_mini_model_config = load_config(less_drunet_mini_model_config_path)
less_drunet_mini_model = get_model(less_drunet_mini_model_config)
less_drunet_mini_model.load_state_dict(torch.load(less_drunet_mini_model_weight_path)['model_state_dict'])
less_drunet_mini_model.to(device)

DRUNet(
  (m_head): AffineConv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
  (m_down): ModuleList(
    (0): Sequential(
      (0): ResBlock(
        (m_res): Sequential(
          (0): AffineConv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
          (1): SortPool()
          (2): AffineConv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
        )
        (sum): ResidualConnection()
      )
      (1): ResBlock(
        (m_res): Sequential(
          (0): AffineConv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
          (1): SortPool()
          (2): AffineConv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
        )
        (sum): ResidualConnection()
      )
      (2): ResBlock(
        (m_res): Sequential(
          (0): AffineConv2d(64, 64, 

In [17]:
summary(less_drunet_mini_model, input_size=(1,in_dim,128,128))

Layer (type:depth-idx)                        Output Shape              Param #
DRUNet                                        [1, 3, 128, 128]          --
├─AffineConv2d: 1-1                           [1, 64, 128, 128]         1,728
├─ModuleList: 1-2                             --                        --
│    └─Sequential: 2-1                        [1, 128, 64, 64]          --
│    │    └─ResBlock: 3-1                     [1, 64, 128, 128]         73,729
│    │    └─ResBlock: 3-2                     [1, 64, 128, 128]         73,729
│    │    └─ResBlock: 3-3                     [1, 64, 128, 128]         73,729
│    │    └─ResBlock: 3-4                     [1, 64, 128, 128]         73,729
│    │    └─AffineConv2d: 3-5                 [1, 128, 64, 64]          32,768
│    └─Sequential: 2-2                        [1, 256, 32, 32]          --
│    │    └─ResBlock: 3-6                     [1, 128, 64, 64]          294,913
│    │    └─ResBlock: 3-7                     [1, 128, 64, 64]     

In [18]:
ordinary_drunet_config_path = '../models/ordinary_drunet/s=0.1/model_config.json'
ordinary_drunet_weight_path = '../models/ordinary_drunet/s=0.1/model.pt'
ordinary_drunet_config = load_config(ordinary_drunet_config_path)
ordinary_drunet_model = get_model(ordinary_drunet_config)
ordinary_drunet_model.load_state_dict(torch.load(ordinary_drunet_weight_path)['model_state_dict'])
ordinary_drunet_model.to(device)

DRUNet(
  (m_head): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (m_down): ModuleList(
    (0): Sequential(
      (0): ResBlock(
        (m_res): Sequential(
          (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (1): ReLU(inplace=True)
          (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
        (sum): ResidualConnection()
      )
      (1): ResBlock(
        (m_res): Sequential(
          (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (1): ReLU(inplace=True)
          (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
        (sum): ResidualConnection()
      )
      (2): ResBlock(
        (m_res): Sequential(
          (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (1): ReLU(inplace=True)
          (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
        (sum):

In [19]:
summary(ordinary_drunet_model, input_size=(1,in_dim,128,128))

Layer (type:depth-idx)                        Output Shape              Param #
DRUNet                                        [1, 3, 128, 128]          --
├─Conv2d: 1-1                                 [1, 64, 128, 128]         1,792
├─ModuleList: 1-2                             --                        --
│    └─Sequential: 2-1                        [1, 128, 64, 64]          --
│    │    └─ResBlock: 3-1                     [1, 64, 128, 128]         73,856
│    │    └─ResBlock: 3-2                     [1, 64, 128, 128]         73,856
│    │    └─ResBlock: 3-3                     [1, 64, 128, 128]         73,856
│    │    └─ResBlock: 3-4                     [1, 64, 128, 128]         73,856
│    │    └─Conv2d: 3-5                       [1, 128, 64, 64]          32,896
│    └─Sequential: 2-2                        [1, 256, 32, 32]          --
│    │    └─ResBlock: 3-6                     [1, 128, 64, 64]          295,168
│    │    └─ResBlock: 3-7                     [1, 128, 64, 64]     

In [20]:
modified_drunet_config_path = '../models/modified_drunet/s=0.1/model_config.json'
modified_drunet_weight_path = '../models/modified_drunet/s=0.1/model.pt'
modified_drunet_config = load_config(modified_drunet_config_path)
modified_drunet_model = get_model(modified_drunet_config)
modified_drunet_model.load_state_dict(torch.load(modified_drunet_weight_path)['model_state_dict'])
modified_drunet_model.to(device)


ModifiedDRUNet(
  (m_head): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (m_down): ModuleList(
    (0): Sequential(
      (0): ResBlock(
        (m_res): Sequential(
          (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): SortPool()
          (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        )
        (sum): ResidualConnection()
      )
      (1): ResBlock(
        (m_res): Sequential(
          (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): SortPool()
          (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        )
        (sum): ResidualConnection()
      )
      (2): ResBlock(
        (m_res): Sequential(
          (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): SortPool()
          (2): Conv2d(64, 64, kernel_size=(3, 3),

In [21]:
summary(modified_drunet_model, input_size=(1,in_dim,128,128))

Layer (type:depth-idx)                        Output Shape              Param #
ModifiedDRUNet                                [1, 3, 128, 128]          --
├─Conv2d: 1-1                                 [1, 64, 128, 128]         1,728
├─ModuleList: 1-2                             --                        --
│    └─Sequential: 2-1                        [1, 128, 64, 64]          --
│    │    └─ResBlock: 3-1                     [1, 64, 128, 128]         73,728
│    │    └─ResBlock: 3-2                     [1, 64, 128, 128]         73,728
│    │    └─ResBlock: 3-3                     [1, 64, 128, 128]         73,728
│    │    └─ResBlock: 3-4                     [1, 64, 128, 128]         73,728
│    │    └─Conv2d: 3-5                       [1, 128, 64, 64]          32,768
│    └─Sequential: 2-2                        [1, 256, 32, 32]          --
│    │    └─ResBlock: 3-6                     [1, 128, 64, 64]          294,912
│    │    └─ResBlock: 3-7                     [1, 128, 64, 64]     